# Agente AI per la Gestione del Rischio Finanziario e Audit Interattivi </p> Contesto Aziendale

L'azienda FinSecure Analytics, specializzata in consulenza finanziaria e gestione del rischio, deve analizzare
costantemente grandi volumi di dati finanziari per valutare esposizioni al rischio, trend di mercato e conformità
normativa. Questo processo richiede un impegno significativo da parte degli analisti e presenta il rischio di omissioni
o interpretazioni errate.

## Obiettivo del Progetto

Sviluppare un agente AI basato su modelli avanzati di NLP e analisi predittiva per supportare il team finanziario nelle
attività di gestione del rischio e audit interattivi, fornendo insight automatici e strumenti di simulazione per scenari
finanziari complessi.

## Soluzione Tecnologica

L'agente AI si basa su tecnologie di Machine Learning e Large Language Models (LLMs), combinando framework come
LangChain e LlamaIndex per offrire le seguenti funzionalità: - Analisi automatizzata dei report finanziari: estrazione
di KPI e identificazione di anomalie. - Simulazione scenari di rischio: modellazione di possibili impatti finanziari in
base ai dati storici e a variabili economiche. - Audit interattivo: chatbot intelligente per rispondere a domande sulle
analisi finanziarie e suggerire azioni correttive. - Dashboard di visualizzazione: grafici e reportistica in tempo reale
per il monitoraggio delle esposizioni al rischio.

## Vantaggi per l'Azienda

📉 Riduzione del Rischio Finanziario

- Identificazione precoce di possibili crisi aziendali.
- Analisi predittiva per prendere decisioni basate sui dati.

⚖️ Maggiore Compliance e Controllo

- Audit automatici per garantire l’aderenza agli standard regolatori.
- Reportistica immediata per semplificare i processi di conformità.

🚀 Ottimizzazione del Lavoro degli Analisti

- Automazione di attività ripetitive e time-consuming.
- Accesso rapido agli insight finanziari con un'interfaccia intuitiva.

## Implementazione

Il progetto prevede tre fasi principali:

1. Fase 1 - Raccolta e Pre-processing Dati: acquisizione e normalizzazione dei dati finanziari aziendali. Questi dati
   possono essere simulati mediante un qualsiasi LLM che genera documenti di testo (es. ChatGPT o Gemini).
2. Fase 2 - Sviluppo dell'Agente AI: addestramento dei modelli e integrazione con gli strumenti
   aziendali.
3. Fase 3 - Testing e Deployment: validazione dei risultati con dati reali e rilascio dell’agente in ambiente
   di produzione.

Per questo progetto è possibile utilizzare dei documenti fittizi, che vanno rilasciati insieme al codice. Il sistema deve
avvalersi delle tecnologie indicate per individuare omissioni, inesattezze, possibili interpretazioni erronee e altri
fattori di rischio, con un approccio conversazionale.

## Conclusioni
L'agente AI di FinSecure Analytics rappresenta una soluzione innovativa per trasformare la gestione del rischio e gli
audit aziendali, riducendo i tempi di analisi, migliorando la precisione dei report e supportando il team finanziario
con strumenti predittivi avanzati.

## Modalità di consegna:
Link pubblico a notebook di Google Colab

## Architettura della soluzione

```
Documenti fittizi ──► LlamaIndex (indice vettoriale) ──┐
Dati storici (tabella) ────────────────────────────────┤
                                                       ▼
                 4 tool ──► Agente LangChain (ReAct + memoria) ◄──► analista (chat)
        cerca_nei_documenti · calcola_kpi · simula_scenario · audit_documento
```

**Come eseguirlo**: crea un file `.env` nella directory del progetto con la tua chiave API Groq e lancia le celle in ordine:

```
GROQ_API_KEY=gsk_...
```

Modelli: `openai/gpt-oss-120b` via ChatGroq (agente e audit), `all-MiniLM-L6-v2` in locale (embedding).


In [1]:
import os
import warnings
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from langchain.agents import create_agent
from langchain.agents.middleware import ClearToolUsesEdit, ContextEditingMiddleware, ModelRetryMiddleware
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

warnings.filterwarnings("ignore")

In [2]:
# Carica la API key dal file .env
load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise EnvironmentError(
        "GROQ_API_KEY non trovata. Creare un file .env con: GROQ_API_KEY=gsk_..."
    )


## Fase 1 — Raccolta e pre-processing dati

Documenti e dati sono **fittizi** (generati per questo progetto) e vengono scritti dal notebook stesso:
- 3 documenti di testo di *Meridiana Logistica S.p.A.*: bilancio sintetico, relazione sui rischi,
  nota integrativa con verbale del comitato audit
- una tabella di dati storici 2020-2024, da cui calcolare i KPI

Nei documenti sono nascosti **5 problemi** (un'inesattezza numerica, un'omissione, una formulazione ambigua,
un'incoerenza tra documenti, un'area di compliance non menzionata): servono per verificare l'agente nella Fase 3.


In [3]:
# Documenti fittizi di "Meridiana Logistica S.p.A." (valori in milioni di euro).
# Contengono 5 problemi volutamente nascosti, che servono a verificare l'agente nella Fase 3.
DOCUMENTI = {
    "bilancio_sintetico_2024.txt": """
MERIDIANA LOGISTICA S.p.A. — BILANCIO SINTETICO ESERCIZIO 2024
(valori in milioni di euro)

1. CONTO ECONOMICO
Ricavi: 171,0 (2023: 162,0; +5,6%)
Costi operativi: 153,0 (2023: 142,0)
EBITDA: 18,0 (2023: 20,0)
Margine EBITDA: 12,3% (2023: 12,3%)
Oneri finanziari netti: 3,6 (tasso medio sul debito: 4,8%)

2. STATO PATRIMONIALE (sintesi)
Debito finanziario netto: 74,0 (2023: 61,0)
Disponibilità liquide: 11,2
Crediti commerciali: 38,5, di cui scaduti oltre 90 giorni: 9,4
Patrimonio netto: 52,0

3. COMMENTO DELLA DIREZIONE
Il margine EBITDA si conferma stabile al 12,3% grazie al controllo dei costi. La crescita dei ricavi
è sostenuta dai nuovi contratti di logistica per l'e-commerce. L'indebitamento aumenta per gli
investimenti nel nuovo polo logistico di Piacenza.
""",
    "relazione_rischi_2024.txt": """
MERIDIANA LOGISTICA S.p.A. — RELAZIONE SUI RISCHI 2024

RISCHIO DI CREDITO
I crediti commerciali scaduti oltre 90 giorni ammontano a 6,1 milioni di euro e sono concentrati su
tre clienti (circa il 55% dell'importo). Il fondo svalutazione crediti è pari a 1,2 milioni.

RISCHIO DI LIQUIDITÀ
Le linee di credito non utilizzate ammontano a 15 milioni. Il finanziamento sindacato da 40 milioni
scade nel 2026 e dovrà essere rinegoziato.

RISCHIO DI TASSO
Una quota prevalente del debito è a tasso variabile. L'esposizione al rischio di tasso è da ritenersi
moderata e adeguatamente presidiata.

RISCHIO DI MERCATO
Il costo del carburante incide per circa il 18% sui costi operativi. Le clausole di adeguamento
prezzo sono presenti solo nel 40% dei contratti con i clienti.

RISCHIO OPERATIVO
L'attività dipende in misura rilevante dal polo logistico di Piacenza, che gestisce la maggior parte
dei volumi.
""",
    "nota_integrativa_audit_2024.txt": """
MERIDIANA LOGISTICA S.p.A. — NOTA INTEGRATIVA E VERBALE DEL COMITATO CONTROLLO E RISCHI
Riunione del 28 marzo 2025

AREE VERIFICATE
Il Comitato ha esaminato: ciclo attivo, ciclo passivo, tesoreria, magazzino e gestione del personale.
Il Comitato ritiene che tutte le aree rilevanti siano state adeguatamente coperte.

COVENANT FINANZIARI
Il finanziamento sindacato prevede un covenant: rapporto tra debito finanziario netto ed EBITDA non
superiore a 4,0x, verificato annualmente al 31 dicembre. Il Comitato ha accertato che il rapporto si
mantiene entro i limiti contrattuali.

CREDITI E PARTI CORRELATE
I crediti scaduti sono sotto controllo. Le operazioni con la società controllata Meridiana Immobiliare
sono state concluse a condizioni di mercato non meglio specificate.

RACCOMANDAZIONI DEL COMITATO
Nessuna.
""",
}

# i documenti vengono salvati su file, così viaggiano insieme al notebook
os.makedirs("documenti", exist_ok=True)
for nome, testo in DOCUMENTI.items():
    with open(f"documenti/{nome}", "w", encoding="utf-8") as f:
        f.write(testo.strip())

print(f"✅ {len(DOCUMENTI)} documenti fittizi salvati in ./documenti")

✅ 3 documenti fittizi salvati in ./documenti


In [4]:
# Dati storici fittizi (milioni di euro) di Meridiana Logistica S.p.A.
storico = pd.DataFrame({
    "anno":            [2020, 2021, 2022, 2023, 2024],
    "ricavi":          [118.0, 131.0, 149.0, 162.0, 171.0],
    "costi_operativi": [104.0, 114.0, 130.0, 142.0, 153.0],
    "debito_netto":    [40.0, 44.0, 52.0, 61.0, 74.0],
    "tasso_medio_pct": [2.5, 2.4, 3.1, 4.2, 4.8],
}).set_index("anno")

# normalizzazione: KPI derivati calcolati sempre con le stesse formule
storico["ebitda"] = storico["ricavi"] - storico["costi_operativi"]
storico["margine_ebitda_pct"] = (storico["ebitda"] / storico["ricavi"] * 100).round(1)
storico["debito_su_ebitda"] = (storico["debito_netto"] / storico["ebitda"]).round(2)

# pre-processing dei documenti: quanti file e quante parole
for nome in sorted(os.listdir("documenti")):
    with open(f"documenti/{nome}", encoding="utf-8") as f:
        print(f"📄 {nome}: {len(f.read().split())} parole")

storico

📄 bilancio_sintetico_2024.txt: 111 parole
📄 nota_integrativa_audit_2024.txt: 115 parole
📄 relazione_rischi_2024.txt: 142 parole


,ricavi,costi_operativi,debito_netto,tasso_medio_pct,ebitda,margine_ebitda_pct,debito_su_ebitda
anno,,,,,,,
2020,118.0,104.0,40.0,2.5,14.0,11.9,2.86
2021,131.0,114.0,44.0,2.4,17.0,13.0,2.59
2022,149.0,130.0,52.0,3.1,19.0,12.8,2.74
2023,162.0,142.0,61.0,4.2,20.0,12.3,3.05
2024,171.0,153.0,74.0,4.8,18.0,10.5,4.11


## Fase 2 — Sviluppo dell'agente

- **LlamaIndex**: indicizza i documenti e recupera i passaggi rilevanti (retrieval con fonte).
- **LangChain**: agente ReAct (`create_agent`) con 4 tool e memoria della conversazione.

| Tool | Cosa fa |
|---|---|
| `cerca_nei_documenti` | recupera passaggi dai documenti (LlamaIndex) |
| `calcola_kpi` | ricalcola i KPI dai dati storici, per verificare i numeri dichiarati |
| `simula_scenario` | what-if su ricavi, costi e tassi + proiezione del trend |
| `audit_documento` | applica la checklist di audit a un singolo documento |


In [5]:
# Indicizzazione con LlamaIndex: chunking + embedding locali + indice vettoriale in memoria
Settings.embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")
Settings.node_parser = SentenceSplitter(chunk_size=300, chunk_overlap=40)

documenti_llama = SimpleDirectoryReader("documenti").load_data()
index = VectorStoreIndex.from_documents(documenti_llama)
retriever = index.as_retriever(similarity_top_k=4)

print(f"✅ Indice pronto — {len(documenti_llama)} documenti, {len(index.docstore.docs)} chunk")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Indice pronto — 3 documenti, 5 chunk


In [6]:
# ── Tool 1: ricerca nei documenti (LlamaIndex) ──
@tool
def cerca_nei_documenti(domanda: str) -> str:
    """Cerca nei documenti aziendali (bilancio, relazione sui rischi, nota integrativa/verbale audit)
    i passaggi rilevanti per la domanda. Restituisce i passaggi con il nome del documento di origine."""
    nodi = retriever.retrieve(domanda)
    if not nodi:
        return "Nessun passaggio rilevante trovato."
    return "\n\n".join(
        f"[Fonte: {n.node.metadata.get('file_name', '?')} | score {(n.score or 0):.2f}]\n{n.node.get_content()}"
        for n in nodi
    )


# test rapido del retrieval
print(cerca_nei_documenti.invoke({"domanda": "crediti scaduti oltre 90 giorni"})[:600])

[Fonte: relazione_rischi_2024.txt | score 0.57]
MERIDIANA LOGISTICA S.p.A. — RELAZIONE SUI RISCHI 2024

RISCHIO DI CREDITO
I crediti commerciali scaduti oltre 90 giorni ammontano a 6,1 milioni di euro e sono concentrati su
tre clienti (circa il 55% dell'importo). Il fondo svalutazione crediti è pari a 1,2 milioni.

RISCHIO DI LIQUIDITÀ
Le linee di credito non utilizzate ammontano a 15 milioni. Il finanziamento sindacato da 40 milioni
scade nel 2026 e dovrà essere rinegoziato.

RISCHIO DI TASSO
Una quota prevalente del debito è a tasso variabile. L'esposizione al rischio di tasso è d


In [7]:
# ── Tool 2: calcolo dei KPI dai dati storici ──
@tool
def calcola_kpi(anno: int) -> str:
    """Calcola i KPI finanziari di un anno (2020-2024) dai dati storici: ricavi, EBITDA, margine EBITDA,
    debito netto/EBITDA, copertura interessi e crescita dei ricavi. Usalo per verificare i numeri
    dichiarati nei documenti."""
    if anno not in storico.index:
        return f"Anno non disponibile. Anni validi: {list(storico.index)}"

    r = storico.loc[anno]
    oneri = r["debito_netto"] * r["tasso_medio_pct"] / 100
    righe = [
        f"KPI {anno} (mln €, dati storici interni):",
        f"- Ricavi: {r['ricavi']:.1f}",
        f"- EBITDA: {r['ebitda']:.1f}",
        f"- Margine EBITDA: {r['margine_ebitda_pct']:.1f}%",
        f"- Debito netto: {r['debito_netto']:.1f}",
        f"- Debito netto/EBITDA: {r['debito_su_ebitda']:.2f}x",
        f"- Copertura interessi (EBITDA/oneri finanziari): {r['ebitda'] / oneri:.2f}x",
    ]
    if anno - 1 in storico.index:
        crescita = (r["ricavi"] / storico.loc[anno - 1, "ricavi"] - 1) * 100
        righe.append(f"- Crescita ricavi vs {anno - 1}: {crescita:+.1f}%")
    return "\n".join(righe)


In [8]:
# ── Tool 3: simulazione di scenari di rischio ──
def calcola_scenario(delta_ricavi_pct=0.0, delta_costi_pct=0.0, delta_tassi_pt=0.0) -> dict:
    """Applica variazioni ai conti 2024 e ricalcola EBITDA, leverage e copertura interessi."""
    b = storico.loc[2024]
    ricavi = b["ricavi"] * (1 + delta_ricavi_pct / 100)
    costi = b["costi_operativi"] * (1 + delta_costi_pct / 100)
    ebitda = ricavi - costi
    oneri = b["debito_netto"] * (b["tasso_medio_pct"] + delta_tassi_pt) / 100
    return {
        "ricavi": ricavi,
        "ebitda": ebitda,
        "oneri_fin": oneri,
        "debito_su_ebitda": b["debito_netto"] / ebitda if ebitda > 0 else np.inf,
        "copertura_interessi": ebitda / oneri,
    }


# scenari predefiniti: (variazione ricavi %, variazione costi %, variazione tassi in punti)
SCENARI = {
    "base": (0, 0, 0),
    "recessione": (-10, -5, 0),
    "rialzo_tassi": (0, 0, 2.0),
    "stress_combinato": (-10, -5, 2.0),
}


def _fmt_x(v: float) -> str:
    return "n/d (EBITDA <= 0)" if np.isinf(v) else f"{v:.2f}x"


@tool
def simula_scenario(scenario: str = "custom", delta_ricavi_pct: float = 0.0,
                    delta_costi_pct: float = 0.0, delta_tassi_pt: float = 0.0) -> str:
    """Simula l'impatto di uno scenario di rischio sui conti 2024 e confrontalo con la situazione base.
    scenario: 'recessione' (ricavi -10%, costi -5%), 'rialzo_tassi' (+2 punti), 'stress_combinato',
    oppure 'custom' per usare i tre parametri (variazioni % di ricavi e costi, variazione dei tassi in
    punti percentuali). Restituisce EBITDA, Debito/EBITDA, copertura interessi e la proiezione 2025
    dell'EBITDA secondo il trend lineare storico."""
    if scenario in SCENARI:
        dr, dc, dt = SCENARI[scenario]
    else:
        dr, dc, dt = delta_ricavi_pct, delta_costi_pct, delta_tassi_pt

    base, sim = calcola_scenario(), calcola_scenario(dr, dc, dt)

    # proiezione 2025: retta di regressione sull'EBITDA storico
    pendenza, intercetta = np.polyfit(storico.index.values, storico["ebitda"].values, 1)
    ebitda_2025 = pendenza * 2025 + intercetta

    return (
        f"Scenario '{scenario}': ricavi {dr:+}%, costi {dc:+}%, tassi {dt:+} punti\n"
        f"- EBITDA: {base['ebitda']:.1f} -> {sim['ebitda']:.1f} mln €\n"
        f"- Debito netto/EBITDA: {_fmt_x(base['debito_su_ebitda'])} -> {_fmt_x(sim['debito_su_ebitda'])}\n"
        f"- Copertura interessi: {base['copertura_interessi']:.2f}x -> {sim['copertura_interessi']:.2f}x\n"
        f"- Proiezione EBITDA 2025 (trend lineare, senza shock): {ebitda_2025:.1f} mln €"
    )

In [9]:
# ── Tool 4: audit di un singolo documento con checklist ──
llm_audit = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

CHECKLIST = """1. OMISSIONI: informazioni che ci si aspetterebbe e che mancano
2. INESATTEZZE: numeri incoerenti o non verificabili
3. AMBIGUITÀ: formulazioni vaghe, senza soglie o dati a supporto
4. INTERPRETAZIONI ERRATE: conclusioni non supportate dai dati presentati
5. COMPLIANCE: obblighi contrattuali (es. covenant) e aree di rischio tipiche non menzionate
   (es. cybersecurity, ESG, continuità operativa)"""


@tool
def audit_documento(nome_documento: str) -> str:
    """Esegue un audit su UN documento applicando la checklist (omissioni, inesattezze, ambiguità,
    interpretazioni errate, compliance). nome_documento: parte del nome del file, es. 'bilancio',
    'relazione_rischi', 'nota_integrativa'. Restituisce un elenco di rilievi in JSON."""
    candidati = [f for f in os.listdir("documenti") if nome_documento.lower() in f.lower()]
    if not candidati:
        return f"Documento non trovato. Disponibili: {os.listdir('documenti')}"

    nome_file = candidati[0]
    with open(f"documenti/{nome_file}", encoding="utf-8") as f:
        testo = f.read()

    prompt = f"""Sei un auditor. Analizza il documento '{nome_file}' con questa checklist:
{CHECKLIST}

DOCUMENTO:
{testo}

Rispondi SOLO con un JSON: una lista di oggetti
{{"tipo": "omissione|inesattezza|ambiguità|interpretazione|compliance", "descrizione": "...", "gravita": "alta|media|bassa"}}"""
    return f"[Documento: {nome_file}]\n" + llm_audit.invoke(prompt).content

In [10]:
SYSTEM_PROMPT = """Sei un assistente di audit e risk management di FinSecure Analytics.
Analizzi i documenti finanziari della società cliente Meridiana Logistica S.p.A. e dialoghi con
l'analista per individuare omissioni, inesattezze, ambiguità, interpretazioni errate e rischi.

STRUMENTI:
- cerca_nei_documenti: recupera passaggi dei documenti (con la fonte)
- calcola_kpi: ricalcola i KPI dai dati storici
- simula_scenario: simula scenari di rischio sui conti 2024
- audit_documento: analizza un singolo documento con la checklist di audit

REGOLE:
1. Non inventare mai numeri: ogni dato deve provenire da uno strumento. Cita sempre il documento di origine.
2. Quando un documento dichiara un KPI, verificalo con calcola_kpi e segnala ogni differenza.
3. Confronta i documenti tra loro (cerca_nei_documenti) per trovare incoerenze.
4. Cerca attivamente omissioni, formulazioni vaghe e obblighi contrattuali (es. covenant) non rispettati o non citati.
5. Se non hai evidenza sufficiente, dillo esplicitamente.

FORMATO: per ogni rilievo indica Rilievo, Gravità (alta/media/bassa), Fonte, Azione correttiva.
Per domande semplici rispondi in modo breve e conversazionale. Rispondi sempre in italiano."""

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

agent = create_agent(
    model=model,
    tools=[cerca_nei_documenti, calcola_kpi, simula_scenario, audit_documento],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),  # memoria della conversazione, per thread_id
    middleware=[
        # il piano gratuito di Groq ha un limite di 8000 token/minuto: se viene superato (errore 429)
        ModelRetryMiddleware(max_retries=5, initial_delay=5, backoff_factor=2, max_delay=60, on_failure="error"),
        # la cronologia viene rimandata al modello a ogni chiamata: gli output dei tool dei turni precedenti
        ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=3000, keep=2)]),
    ],
)
print("✅ Agente pronto")

✅ Agente pronto


In [11]:
def chat(domanda: str, thread_id: str = "audit_001") -> str:
    """Invia una domanda all'agente (la memoria è legata al thread_id) e stampa risposta e tool usati."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": domanda}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    msgs = result["messages"]

    # tool chiamati nel turno corrente (dopo l'ultimo messaggio dell'utente)
    ultimo_utente = max(i for i, m in enumerate(msgs) if m.type == "human")
    tool_usati = [tc["name"] for m in msgs[ultimo_utente:] if m.type == "ai" for tc in m.tool_calls]
    risposta = msgs[-1].content

    print(f"\n{'='*70}\n👤 {domanda}\n🔧 Tool usati: {tool_usati or 'nessuno'}\n{'-'*70}")
    print(f"🤖 {risposta}")
    return risposta


## Fase 3 — Testing

Conversazione di 8 turni che copre: KPI, verifica numerica, incoerenze tra documenti, audit di un documento,
simulazione di scenari e domande di approfondimento (che mettono alla prova la memoria).
Poiché i documenti sono fittizi, sappiamo in anticipo quali problemi sono stati inseriti:
possiamo quindi controllare quanti ne vengono individuati.


In [ ]:
DOMANDE = [
    "Ciao! Devo preparare un audit su Meridiana Logistica. Quali sono i KPI principali del 2024?",
    "Il margine EBITDA dichiarato nel bilancio corrisponde ai dati storici? Verifica.",
    "Ci sono incoerenze tra bilancio e relazione sui rischi? Controlla anche il rispetto dei covenant sul debito.",
    "Fai un audit della nota integrativa e verbale del comitato: cosa manca o è poco chiaro?",
    "Cosa succede ai conti se i tassi salgono di 2 punti?",
    "E se i ricavi calano del 15% con costi operativi in calo del 5%?",
    "Tra tutti i rilievi emersi, qual è il più grave? Che azioni correttive suggerisci?",
    "Riepiloga tutti i rilievi emersi in ordine di gravità.",
]

risposte = []
for domanda in DOMANDE:
    risposte.append(chat(domanda))


In [ ]:
# Anomalie volutamente inserite nei documenti: l'agente le ha trovate?
# Ogni anomalia è "trovata" se nelle risposte compaiono tutte le keyword di almeno un gruppo
ANOMALIE = {
    "A1 Margine EBITDA dichiarato (12,3%) ≠ calcolato (10,5%)": [["10.5"]],
    "A2 Covenant Debito/EBITDA ≤ 4,0x violato (4,1x) ma non segnalato": [["covenant", "4.1"]],
    "A3 'Esposizione al rischio di tasso moderata' senza soglie": [["moderat"]],
    "A4 Crediti scaduti: 9,4 mln (bilancio) vs 6,1 mln (rischi)": [["9.4", "6.1"]],
    "A5 Nessun cenno a cybersecurity / continuità operativa": [["cyber"], ["sicurezza informatica"], ["nis2"]],
}

# normalizza: minuscolo e virgola decimale -> punto
testo = " ".join(risposte).lower().replace(",", ".")

print(f"{'Anomalia nascosta':<68} Trovata")
print("-" * 76)
trovate = 0
for nome, gruppi in ANOMALIE.items():
    ok = any(all(k in testo for k in g) for g in gruppi)
    trovate += ok
    print(f"{nome:<68} {'✅' if ok else '❌'}")
print(f"\nAnomalie individuate: {trovate}/{len(ANOMALIE)}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# 1) Trend storico del leverage rispetto al covenant
ax1.plot(storico.index, storico["debito_su_ebitda"], marker="o", color="#4C78A8")
ax1.axhline(4.0, color="#E45756", linestyle="--", linewidth=1)
ax1.text(2020, 4.05, "covenant 4,0x", color="#E45756", fontsize=9)
for anno, val in storico["debito_su_ebitda"].items():
    ax1.annotate(f"{val:.2f}x", (anno, val), textcoords="offset points",
                 xytext=(0, 8), ha="center", fontsize=9)
ax1.set_xticks(storico.index)
ax1.set_ylim(2, 4.8)
ax1.set_title("Debito netto / EBITDA")
ax1.set_ylabel("multiplo (x)")

# 2) EBITDA 2024 negli scenari di rischio
nomi = list(SCENARI)
risultati_scenari = [calcola_scenario(*SCENARI[n]) for n in nomi]
ax2.bar(nomi, [r["ebitda"] for r in risultati_scenari], color="#4C78A8")
for i, r in enumerate(risultati_scenari):
    lev = "n/d" if np.isinf(r["debito_su_ebitda"]) else f"{r['debito_su_ebitda']:.1f}x"
    ax2.text(i, r["ebitda"] + 0.3, f"{r['ebitda']:.1f} mln\nD/E {lev}",
             ha="center", fontsize=9)
ax2.set_ylim(0, 24)
ax2.set_title("EBITDA 2024 per scenario")
ax2.set_ylabel("mln €")

plt.tight_layout()
plt.show()

## Conclusioni e limiti

**Cosa abbiamo costruito**
- Un agente conversazionale (LangChain `create_agent` + memoria `InMemorySaver`) con 4 strumenti:
  ricerca documentale (LlamaIndex), calcolo KPI, simulazione di scenari e audit di un documento.
- Un set di documenti fittizi con 5 problemi nascosti, utile per verificare in modo oggettivo
  se l'agente individua omissioni, inesattezze, ambiguità e incoerenze.

**Limiti**
- I dati sono **fittizi** e molto semplificati: la validazione con dati reali (Fase 3 della traccia)
  richiede documenti aziendali veri e la revisione di un analista.
- Gli scenari sono what-if deterministici e la proiezione è un trend lineare: non è un modello predittivo.
- La memoria vive solo nella sessione del notebook; il "deployment" è il notebook Colab stesso.
- L'agente è un supporto all'analista: i rilievi vanno sempre verificati sulle fonti citate.

**Possibili sviluppi**: più documenti e KPI, persistenza dell'indice (ChromaDB), un'interfaccia
(Gradio/Streamlit), un grafo LangGraph per un audit strutturato in più passaggi.
